### Healthcare sales - revenue and regional margins analysis
Pulls sales data from MySQL, applies a QC/governance layer for four defect types, computes revenue/margin/discount-leakage metrics by county and segment, and identifies loss-making transactions.

In [1]:
# Imports and connection

import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
db_user = os.getenv("DB_USER")
db_password = quote_plus(os.getenv("DB_PASSWORD"))
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")
print("Connected.")

Connected.


#### 1. Extract sales via SQL (join fact against both dimensions)

In [2]:
QUERY = """
SELECT f.sales_id, f.invoice_date, f.client_key, c.client_segment, c.county,
       f.product_key, p.product_name, p.category, p.unit_cost, p.unit_price,
       f.quantity_ordered, f.invoiced_unit_price
FROM fact_sales f
INNER JOIN dim_clients c ON f.client_key = c.client_key
INNER JOIN dim_products p ON f.product_key = p.product_key;
"""
df = pd.read_sql(QUERY, engine)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
print(f"Rows extracted: {len(df)}")
df.head()

Rows extracted: 483


,sales_id,invoice_date,client_key,client_segment,county,product_key,product_name,category,unit_cost,unit_price,quantity_ordered,invoiced_unit_price
0,20002,2026-02-04,5006,Private Hospital,Mombasa,101,Amoxicillin 500mg,Pharmaceutical,50.0,80.0,106,78.10
1,20013,2026-02-24,5014,Public Hospital,Uasin Gishu,101,Amoxicillin 500mg,Pharmaceutical,50.0,80.0,216,69.56
2,20023,2026-02-04,5011,Pharmacy,Nakuru,101,Amoxicillin 500mg,Pharmaceutical,50.0,80.0,145,78.72
3,20025,2026-06-03,5001,Public Hospital,Nairobi,101,Amoxicillin 500mg,Pharmaceutical,50.0,80.0,4947,78.94
4,20026,2026-02-28,5003,Private Hospital,Nairobi,101,Amoxicillin 500mg,Pharmaceutical,50.0,80.0,60,78.06


#### 2. QC layer

In [3]:
QTY_OUTLIER_THRESHOLD = 400  # verified: genuine max=290, smallest injected outlier=442

RULES = {
    'negative_quantity': df['quantity_ordered'] < 0,
    'zero_price': df['invoiced_unit_price'] == 0,
    'price_above_list': df['invoiced_unit_price'] > df['unit_price'],
    f'quantity_outlier_over_{QTY_OUTLIER_THRESHOLD}': df['quantity_ordered'] > QTY_OUTLIER_THRESHOLD,
}

audit_rows = []
for rule_name, mask in RULES.items():
    for sid in df.loc[mask, 'sales_id']:
        audit_rows.append({'sales_id': sid, 'exclusion_reason': rule_name})

audit_log = pd.DataFrame(audit_rows).sort_values('sales_id').reset_index(drop=True)
excluded_ids = set(audit_log['sales_id'])
clean_df = df[~df['sales_id'].isin(excluded_ids)].copy()

assert len(excluded_ids) + len(clean_df) == len(df)
assert (clean_df['quantity_ordered'] > 0).all()
assert (clean_df['invoiced_unit_price'] > 0).all()
assert (clean_df['invoiced_unit_price'] <= clean_df['unit_price']).all()
assert (clean_df['quantity_ordered'] <= QTY_OUTLIER_THRESHOLD).all()

print(f"Flagged: {len(excluded_ids)}  |  Clean: {len(clean_df)}")
audit_log['exclusion_reason'].value_counts()

Flagged: 53  |  Clean: 430


exclusion_reason
quantity_outlier_over_400    17
zero_price                   13
price_above_list             12
negative_quantity            11
Name: count, dtype: int64

#### 3. Overall KPI metrics

In [4]:
clean_df['revenue'] = clean_df['quantity_ordered'] * clean_df['invoiced_unit_price']
clean_df['list_price_revenue'] = clean_df['quantity_ordered'] * clean_df['unit_price']
clean_df['cost'] = clean_df['quantity_ordered'] * clean_df['unit_cost']

total_revenue = clean_df['revenue'].sum()
total_cost = clean_df['cost'].sum()
list_price_revenue = clean_df['list_price_revenue'].sum()
gross_margin_pct = (total_revenue - total_cost) / total_revenue
discount_leakage = list_price_revenue - total_revenue
discount_leakage_pct = discount_leakage / list_price_revenue
total_units_sold = clean_df['quantity_ordered'].sum()
active_clients = clean_df['client_key'].nunique()

print(f"Total Revenue:       {total_revenue:,.2f}")
print(f"Total Cost:          {total_cost:,.2f}")
print(f"Gross Margin %:      {gross_margin_pct:.2%}")
print(f"Discount Leakage:    {discount_leakage:,.2f}  ({discount_leakage_pct:.2%})")
print(f"Total Units Sold:    {total_units_sold:,}")
print(f"Active Clients:      {active_clients}")

Total Revenue:       12,865,041.01
Total Cost:          8,995,980.00
Gross Margin %:      30.07%
Discount Leakage:    956,998.99  (6.92%)
Total Units Sold:    38,412
Active Clients:      15


#### 4. Breakdown by county (Treemap source) and segment (Matrix source)

In [5]:
by_county = clean_df.groupby('county').apply(lambda g: pd.Series({
    'n': len(g),
    'revenue': g['revenue'].sum(),
    'discount_leakage_pct': (g['list_price_revenue'].sum() - g['revenue'].sum()) / g['list_price_revenue'].sum(),
    'gross_margin_pct': (g['revenue'].sum() - g['cost'].sum()) / g['revenue'].sum(),
}), include_groups=False).round(4)

by_segment = clean_df.groupby('client_segment').apply(lambda g: pd.Series({
    'revenue': g['revenue'].sum(),
    'cost': g['cost'].sum(),
    'gross_margin_pct': (g['revenue'].sum() - g['cost'].sum()) / g['revenue'].sum(),
}), include_groups=False).round(2)

assert abs(clean_df['revenue'].sum() - by_county['revenue'].sum()) < 1

print(by_county)
print()
print(by_segment)

                n     revenue  discount_leakage_pct  gross_margin_pct
county                                                               
Kisumu       95.0  2753903.28                0.0640            0.3056
Mombasa      68.0  2248013.03                0.0488            0.3164
Nairobi      88.0  2265423.62                0.0904            0.2874
Nakuru       90.0  2706519.95                0.0780            0.2915
Uasin Gishu  89.0  2891181.13                0.0644            0.3030

                     revenue       cost  gross_margin_pct
client_segment                                           
Pharmacy          3577595.50  2546510.0              0.29
Private Hospital  4550520.36  3166870.0              0.30
Public Hospital   4736925.15  3282600.0              0.31


#### 5. Loss-making transactions (excessive discounting eroding below cost)

In [6]:
clean_df['margin_pct'] = (clean_df['revenue'] - clean_df['cost']) / clean_df['revenue']
loss_making = clean_df[clean_df['margin_pct'] < 0]

print(f"Loss-making transactions: {len(loss_making)} ({len(loss_making)/len(clean_df)*100:.1f}%)")
print(f"Revenue involved: {loss_making['revenue'].sum():,.2f}")
print(f"Net loss (KES): {(loss_making['revenue'] - loss_making['cost']).sum():,.2f}")
print("\nBy county:\n", loss_making['county'].value_counts())
print("\nBy segment:\n", loss_making['client_segment'].value_counts())

Loss-making transactions: 15 (3.5%)
Revenue involved: 305,793.42
Net loss (KES): -23,176.58

By county:
 county
Kisumu         6
Nakuru         5
Nairobi        3
Uasin Gishu    1
Name: count, dtype: int64

By segment:
 client_segment
Private Hospital    6
Pharmacy            6
Public Hospital     3
Name: count, dtype: int64


#### 6. Category split (bar chart source) and monthly trend (line chart source)

In [7]:
by_category = clean_df.groupby('category')['revenue'].sum()
print(by_category)

clean_df['month'] = clean_df['invoice_date'].dt.to_period('M')
monthly = clean_df.groupby('month')['revenue'].sum()
print(monthly)

category
Pharmaceutical    6745217.64
Surgical          6119823.37
Name: revenue, dtype: float64
month
2026-01    1815345.59
2026-02    2122041.65
2026-03    2945013.32
2026-04    1624458.72
2026-05    2233730.29
2026-06    2124451.44
Freq: M, Name: revenue, dtype: float64


In [10]:
export_cols = ['sales_id', 'invoice_date', 'client_key', 'product_key', 'quantity_ordered', 'invoiced_unit_price']
clean_df[export_cols].to_csv('../outputs/clean_sales.csv', index=False)
audit_log.to_csv('../outputs/sales_audit_log.csv', index=False)
    
assert len(pd.read_csv('../outputs/clean_sales.csv')) == len(clean_df)
assert len(pd.read_csv('../outputs/sales_audit_log.csv')) == len(audit_log)
print("Exports verified: row counts match.")

Exports verified: row counts match.
